# YOLOv8 Detection Modules

**Purpose:** Shared modules for all YOLOv8 detection experiments.

**Usage:** Run this notebook first, then use `%run ./YOLOv8_modules.ipynb` in experiment notebooks.

## Cell 1: Imports & Setup

In [ ]:
# ============================================================================
# Install required dependencies (if not already installed)
# ============================================================================
# This cell will install dependencies if they are missing.
# It checks for 'ultralytics' as a proxy for all required packages.
# ============================================================================

import sys
import warnings
warnings.filterwarnings('ignore')

try:
    import ultralytics
    print("✓ ultralytics already installed")
except ImportError:
    print("Installing required dependencies...")
    print("This may take a few minutes...")
    !{sys.executable} -m pip install ultralytics opencv-python matplotlib seaborn pyyaml tqdm --quiet
    print("✓ Dependencies installed successfully")
    import ultralytics  # Import after installation

# Now import all required packages
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from ultralytics.nn.modules import C2f
import yaml
import cv2
import copy
from typing import Dict, List, Optional
from tqdm import tqdm

%matplotlib inline

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('Warning: CUDA not available, using CPU')

# Print package versions
print(f'\nPackage Versions:')
print(f'  PyTorch: {torch.__version__}')
try:
    print(f'  Ultralytics: {ultralytics.__version__}')
except NameError:
    print('  Ultralytics: Not available')
print(f'  OpenCV: {cv2.__version__}')
print(f'  NumPy: {np.__version__}')

## Cell 2: YOLOv8Detector Model

In [ ]:
class YOLOv8Detector(nn.Module):
    """
    YOLOv8 detector with configurable backbone and customization support.
    """
    
    def __init__(self, backbone: str = 'm', input_size: int = 640, 
                 confidence_threshold: float = 0.5, nms_iou_threshold: float = 0.45,
                 pretrained: bool = True, customize_type: Optional[str] = None):
        """
        Initialize YOLOv8 detector.
        
        Args:
            backbone: Model size ('n', 's', 'm', 'l', 'x')
            input_size: Input image size
            confidence_threshold: Detection confidence threshold
            nms_iou_threshold: NMS IoU threshold
            pretrained: Use pretrained weights
            customize_type: Type of customization ('deeper', 'shallower', or None)
        """
        import os
        
        # Disable MLflow globally before loading model
        os.environ['MLFLOW_DISABLE'] = 'true'
        
        super(YOLOv8Detector, self).__init__()
        
        self.backbone = backbone
        self.input_size = input_size
        self.confidence_threshold = confidence_threshold
        self.nms_iou_threshold = nms_iou_threshold
        self.customize_type = customize_type
        
        # Load YOLOv8 model
        model_name = f'yolov8{backbone}.pt' if pretrained else f'yolov8{backbone}.yaml'
        self.model = YOLO(model_name)
        
        # Remove MLflow callbacks after model initialization
        if hasattr(self.model, 'callbacks'):
            # Clear all MLflow-related callbacks
            for event in list(self.model.callbacks.keys()):
                self.model.callbacks[event] = [
                    cb for cb in self.model.callbacks[event]
                    if 'mlflow' not in str(cb).lower() and 'MLFlow' not in str(cb)
                ]
        
        # Apply customization after loading
        if customize_type == 'deeper':
            self._add_conv_layers()
        elif customize_type == 'shallower':
            self._reduce_conv_layers()
        
        # Set thresholds
        self.model.model.conf = confidence_threshold
        self.model.model.iou = nms_iou_threshold
    
    def _add_conv_layers(self):
        """
        V2: Add extra convolutional layers to the backbone after layer2.
        Purpose: Deepen shallow-layer feature extraction.
        """
        try:
            device = next(self.model.model.parameters()).device
            
            # Create new Conv layers (after backbone index 2)
            new_conv_1 = nn.Sequential(
                nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.SiLU(inplace=True)
            ).to(device)
            
            new_conv_2 = nn.Sequential(
                nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.SiLU(inplace=True)
            ).to(device)
            
            # Store original layers
            original_layers = list(self.model.model.model.children())
            
            # Create new model with inserted layers
            new_layers = []
            
            # Copy layers 0-2 (keep as is)
            for i in range(3):
                new_layers.append(original_layers[i])
            
            # Add new conv layers
            new_layers.append(new_conv_1)
            new_layers.append(new_conv_2)
            
            # Copy remaining layers
            for i in range(3, len(original_layers)):
                new_layers.append(original_layers[i])
            
            # Replace the model's sequential with new layers
            self.model.model.model = nn.Sequential(*new_layers)
            
            print("✅ Added 2 convolutional layers to backbone (deeper model)")
            
        except Exception as e:
            print(f"️ Warning: Could not add conv layers: {e}")
            print("Continuing with standard model...")
    
    def _reduce_conv_layers(self):
        """
        V3: Reduce convolutional layers in the backbone by modifying C2f module.
        Purpose: Create a lighter model with faster inference.
        """
        try:
            backbone = self.model.model.model
            
            # Target C2f module at backbone index 6 (P4/16 level)
            target_idx = 6
            target_layer = backbone[target_idx]
            
            if not isinstance(target_layer, C2f):
                raise ValueError(f"Expected C2f at backbone index {target_idx}, found {type(target_layer)}")
            
            c1 = target_layer.cv1.conv.in_channels
            c2 = target_layer.cv2.conv.out_channels
            original_n = len(target_layer.m)
            
            # Compute expansion ratio
            e = target_layer.cv1.conv.out_channels / (2 * c2)
            shortcut = getattr(target_layer.m[0], 'shortcut', False)
            g = target_layer.cv1.conv.groups
            
            # New reduced repeat count: half the original
            new_n = max(1, original_n // 2)
            reduced_layer = C2f(c1, c2, n=new_n, shortcut=shortcut, g=g, e=e)
            
            # Replace the layer in-place
            backbone[target_idx] = reduced_layer
            self.model.model.model = backbone
            
            print(f"✅ Replaced C2f at backbone index {target_idx}: original n={original_n}, new n={new_n}")
            print("✅ Shallower backbone configured successfully")
            
        except Exception as e:
            print(f"️ Warning: Could not reduce conv layers: {e}")
            print("Continuing with standard model...")
    
    def forward(self, x, **kwargs):
        """Forward pass."""
        return self.model(x, verbose=False, **kwargs)
    
    def train_model(self, data: str, epochs: int = 100, imgsz: int = None, **kwargs):
        """
        Train the model.
        
        Args:
            data: Dataset config path
            epochs: Number of epochs
            imgsz: Image size
            **kwargs: Additional training arguments
        """
        if imgsz is None:
            imgsz = self.input_size
        
        results = self.model.train(
            data=data,
            epochs=epochs,
            imgsz=imgsz,
            verbose=True,
            **kwargs
        )
        return results
    
    def save(self, save_path: str):
        """Save model."""
        self.model.save(save_path)

print("✓ YOLOv8Detector defined")

## Cell 3: Model Configurations

In [ ]:
# Model configurations
YOLOV8_BASELINE_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': None  # Baseline - no customization
}

YOLOV8_V2_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': 'deeper'  # Adds conv layers to backbone
}

YOLOV8_V3_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': 'shallower'  # Reduces conv layers in backbone
}

print("✓ Model configurations defined")

## Cell 4: YOLOv8Trainer

In [ ]:
class YOLOv8Trainer:
    """
    Simple trainer for YOLOv8 models using Ultralytics framework.
    """
    
    def __init__(self, learning_rate: float = 0.001, batch_size: int = 16,
                 epochs: int = 100, optimizer: str = 'adam', 
                 weight_decay: float = 1e-4, use_amp: bool = True,
                 patience: int = 15, cos_lr: bool = False, close_mosaic: int = 0):
        """
        Initialize YOLOv8 trainer.
        
        Args:
            learning_rate: Initial learning rate
            batch_size: Batch size
            epochs: Number of epochs
            optimizer: Optimizer type ('adam', 'sgd', 'adamw')
            weight_decay: L2 regularization
            use_amp: Use mixed precision training
            patience: Early stopping patience
            cos_lr: Use cosine learning rate scheduler
            close_mosaic: Stop mosaic augmentation N epochs before end
        """
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.optimizer = optimizer
        self.weight_decay = weight_decay
        self.use_amp = use_amp
        self.patience = patience
        self.cos_lr = cos_lr
        self.close_mosaic = close_mosaic
    
    def train(self, model, train_data: str, val_data: str, 
             output_dir: str, **kwargs):
        """
        Train YOLOv8 model.
        
        Args:
            model: YOLOv8Detector instance
            train_data: Training dataset config path
            val_data: Validation dataset config path
            output_dir: Directory to save outputs
            **kwargs: Additional training arguments
        
        Returns:
            Training results object from Ultralytics
        """
        print(f'\nTraining Configuration:')
        print(f'  Learning Rate: {self.learning_rate}')
        print(f'  Batch Size: {self.batch_size}')
        print(f'  Epochs: {self.epochs}')
        print(f'  Optimizer: {self.optimizer}')
        print(f'  Weight Decay: {self.weight_decay}')
        print(f'  Mixed Precision: {self.use_amp}')
        print(f'  Early Stopping Patience: {self.patience}')
        print(f'  Cosine LR: {self.cos_lr}')
        print(f'  Close Mosaic: {self.close_mosaic} epochs before end\n')
        
        # Train using Ultralytics framework
        results = model.train_model(
            data=train_data,
            epochs=self.epochs,
            batch=self.batch_size,
            lr0=self.learning_rate,
            optimizer=self.optimizer,
            weight_decay=self.weight_decay,
            amp=self.use_amp,
            patience=self.patience,
            cos_lr=self.cos_lr,
            close_mosaic=self.close_mosaic,
            project=str(Path(output_dir).absolute()),
            name='train',
            exist_ok=True,
            **kwargs
        )
        
        return results

print("✓ YOLOv8Trainer defined")

## Cell 5: Training Configurations

In [ ]:
# Training configurations for each experiment
YOLOV8_V1_CONFIG = {
    # Baseline Configuration
    'learning_rate': 0.001,
    'batch_size': 16,
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 1e-4,
    'use_amp': True,
    'patience': 50,
    'cos_lr': False,
    'close_mosaic': 0
}

YOLOV8_V2_CONFIG = {
    # Deeper Backbone Configuration
    'learning_rate': 0.0005,  # Lower LR for deeper model stability
    'batch_size': 12,         # Smaller batch due to larger model
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 5e-4,     # Higher weight decay to prevent overfitting
    'use_amp': True,
    'patience': 50,
    'cos_lr': True,           # Cosine LR schedule for better convergence
    'close_mosaic': 10        # Close mosaic in last 10 epochs
}

YOLOV8_V3_CONFIG = {
    # Shallower Backbone Configuration
    'learning_rate': 0.001,
    'batch_size': 20,         # Larger batch possible due to smaller model
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 1e-4,
    'use_amp': True,
    'patience': 50,
    'cos_lr': False,
    'close_mosaic': 0
}

print("✓ Training configurations defined")

In [ ]:
class DetectionEvaluator:
    """
    Evaluator for YOLOv8 detection models.
    """
    
    def __init__(self):
        self.metrics = {}
        self.last_val_results = None
    
    def evaluate_yolov8(self, model, test_dataset, output_dir=None):
        """
        Evaluate YOLOv8 model on test dataset.
        
        Args:
            model: YOLOv8Detector instance
            test_dataset: Test dataset config path
            output_dir: Optional directory to save evaluation metadata
        
        Returns:
            dict: Evaluation metrics including mAP, precision, recall, F1
        """
        print("=" * 80)
        print("MODEL EVALUATION")
        print("=" * 80)
        
        # Use Ultralytics built-in validation
        if output_dir is not None:
            eval_path = Path(output_dir) / 'evaluation'
            eval_path.mkdir(parents=True, exist_ok=True)
            results = model.model.val(
                data=test_dataset,
                split='test',
                project=str(eval_path),
                name='val',
                save_json=False,
                save_hybrid=False,
                verbose=True
            )
        else:
            results = model.model.val(
                data=test_dataset,
                split='test',
                save_json=False,
                save_hybrid=False,
                verbose=True
            )
        self.last_val_results = results
        
        # Extract metrics - handle different data types (numpy arrays, tensors, scalars)
        def safe_float(value, default=0.0):
            """Safely convert value to float, handling arrays and tensors."""
            if value is None:
                return default
            try:
                # Try direct conversion first
                return float(value)
            except (TypeError, ValueError):
                try:
                    # If it's an array/tensor, extract the scalar value
                    import numpy as np
                    if hasattr(value, 'item'):
                        return float(value.item())
                    elif hasattr(value, 'numpy'):
                        return float(value.numpy())
                    elif isinstance(value, (list, tuple, np.ndarray)):
                        return float(value[0]) if len(value) > 0 else default
                except:
                    pass
            return default
        
        try:
            # Ultralytics 8.x style
            map50 = safe_float(results.box.map50)
            map50_95 = safe_float(results.box.map)
            precision = safe_float(results.box.mp)
            recall = safe_float(results.box.mr)
            # f1 = safe_float(results.box.f1)   not available in some versions, compute manually
            f1 = (2 * precision * recall) / (precision + recall + 1e-16)
        except AttributeError:
            # Fallback for different Ultralytics versions
            map50 = safe_float(results.metrics.get('map50', 0))
            map50_95 = safe_float(results.metrics.get('map', 0))
            precision = safe_float(results.metrics.get('precision', 0))
            recall = safe_float(results.metrics.get('recall', 0))
            # f1 = safe_float(results.metrics.get('f1', 0))   not available in some versions, compute manually
            f1 = (2 * precision * recall) / (precision + recall + 1e-16)
        
        metrics = {
            'map50': map50,
            'map50_95': map50_95,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
        
        self.metrics = metrics
        
        if output_dir is not None:
            eval_dir = Path(output_dir) / 'evaluation'
            eval_dir.mkdir(parents=True, exist_ok=True)
            try:
                import json
                with open(eval_dir / 'metrics.json', 'w') as f:
                    json.dump(metrics, f, indent=2)
                if hasattr(results, 'confusion_matrix'):
                    cm = results.confusion_matrix.matrix
                    with open(eval_dir / 'confusion_matrix.json', 'w') as f:
                        json.dump({'confusion_matrix': cm.tolist()}, f, indent=2)
                print(f"✓ Saved evaluation metadata to {eval_dir}")
            except Exception as e:
                print(f"Warning: Could not save evaluation metadata: {e}")
        
        print(f'\n=== Test Set Metrics ===')
        print(f"mAP@0.5: {metrics['map50']:.4f}")
        print(f"mAP@0.5:0.95: {metrics['map50_95']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1-Score: {metrics['f1']:.4f}")
        
        return metrics
    
    def plot_detection_comparison(self, output_dir=None):
        """
        Plot detection comparison images (labels vs predictions) from evaluation results.
        Displays only 1 pair: val_batch0_labels.jpg vs val_batch0_pred.jpg (vertical layout).
        
        Args:
            output_dir: Directory containing evaluation results
            
        Returns:
            matplotlib figure
        """
        import matplotlib.image as mpimg
        
        if output_dir is None:
            print("Warning: output_dir not provided")
            return None
        
        # Look for val_batch comparison images in evaluation directory
        eval_dir = Path(output_dir) / 'evaluation' / 'val'
        if not eval_dir.exists():
            # Try alternative path
            eval_dir = Path(output_dir) / 'train' / 'val'
        
        if not eval_dir.exists():
            print(f"Warning: Evaluation directory not found at {eval_dir}")
            return None
        
        # Find label/pred image pairs
        label_images = sorted(eval_dir.glob('val_batch*_labels.jpg'))
        pred_images = sorted(eval_dir.glob('val_batch*_pred.jpg'))
        
        if not label_images or not pred_images:
            print(f"Warning: No detection comparison images found in {eval_dir}")
            print(f"  Label images: {len(label_images)}")
            print(f"  Pred images: {len(pred_images)}")
            return None
        
        print(f"Found {len(label_images)} comparison pairs, displaying Batch 0:")
        print(f"  {label_images[0].name} vs {pred_images[0].name}")
        
        # Display only 1 pair (vertical layout)
        fig, axes = plt.subplots(2, 1, figsize=(12, 12))
        
        # Load and display ground truth labels (top)
        img_label = mpimg.imread(label_images[0])
        axes[0].imshow(img_label)
        axes[0].set_title('Ground Truth Labels (Batch 0)', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        # Load and display predictions (bottom)
        img_pred = mpimg.imread(pred_images[0])
        axes[1].imshow(img_pred)
        axes[1].set_title('Model Predictions (Batch 0)', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        return fig
    
    def plot_detection_results(self, model, test_dataset, num_samples=5):
        """
        Plot detection results with bounding boxes inline.
        
        Args:
            model: YOLOv8Detector instance
            test_dataset: Dataset config path
            num_samples: Number of sample images to display
        
        Returns:
            matplotlib figure
        """
        # Load dataset config to get test image paths
        with open(test_dataset, 'r') as f:
            config = yaml.safe_load(f)
        
        print(f"Dataset config loaded: {test_dataset}")
        print(f"Config keys: {list(config.keys())}")
        
        # Try different possible keys for test/val path
        test_path = config.get('test', config.get('val', config.get('path', '')))
        print(f"Test/val path from config: {test_path}")
        
        if not test_path:
            print("Warning: Test path not found in dataset config")
            print(f"Available keys: {list(config.keys())}")
            return None
        
        # Handle relative paths
        config_dir = Path(test_dataset).parent
        test_dir = Path(test_path)
        if not test_dir.is_absolute():
            # Fix: ../test/images -> test/images (relative to yolo/)
            if test_path.startswith('../'):
                test_dir = (config_dir / test_path[3:]).resolve()
            else:
                test_dir = (config_dir / test_path).resolve()
            print(f"Test directory: {test_dir}")
        
        if not test_dir.exists() or not test_dir.is_dir():
            print(f"Error: Test directory not found: {test_dir}")
            return None
        
        print(f"Using test directory: {test_dir}")
        
        # Get image files
        if test_dir.is_file():
            # If it's a txt file with image paths
            print(f"Reading image paths from: {test_dir}")
            with open(test_dir, 'r') as f:
                image_paths = [line.strip() for line in f.readlines()[:num_samples]]
        else:
            # If it's a directory
            print(f"Scanning directory: {test_dir}")
            image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
            image_paths = []
            for ext in image_extensions:
                found = list(test_dir.rglob(ext))
                image_paths.extend(found)
                print(f"  Found {len(found)} {ext} files")
            image_paths = [str(p) for p in image_paths[:num_samples]]
        
        print(f"Total images found: {len(image_paths)}")
        
        if not image_paths:
            print("Warning: No images found for visualization")
            return None
        
        # Run inference and plot
        print(f"Displaying {min(len(image_paths), num_samples)} detection samples...")
        fig, axes = plt.subplots(1, min(len(image_paths), num_samples), figsize=(20, 5))
        if min(len(image_paths), num_samples) == 1:
            axes = [axes]
        
        for idx, img_path in enumerate(image_paths[:num_samples]):
            print(f"  Processing: {img_path}")
            # Run inference
            results = model.model(img_path, verbose=False)
            
            # Get the first result
            result = results[0]
            
            # Read image
            img = cv2.imread(img_path)
            if img is None:
                print(f"    Warning: Could not read image {img_path}")
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Plot on image
            ax = axes[idx]
            ax.imshow(img)
            
            # Draw bounding boxes
            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf = box.conf[0].cpu().numpy()
                    cls = int(box.cls[0].cpu().numpy())
                    
                    # Draw rectangle
                    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                        fill=False, color='red', linewidth=2)
                    ax.add_patch(rect)
                    
                    # Add label
                    class_name = result.names[cls] if hasattr(result, 'names') else f"Class {cls}"
                    label = f"{class_name}: {conf:.2f}"
                    ax.text(x1, y1-5, label, color='red', fontsize=10,
                           bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
            else:
                print(f"    No detections found in this image")
            
            ax.set_title(f"Sample {idx+1}", fontsize=12)
            ax.axis('off')
        
        plt.tight_layout()
        return fig
    
    def extract_results_csv(self, output_dir):
        """
        Extract training data from results.csv.
        
        Args:
            output_dir: Directory containing the train/ subdirectory
            
        Returns:
            pandas.DataFrame or None
        """
        csv_path = Path(output_dir) / 'train' / 'results.csv'
        if not csv_path.exists():
            print(f"Warning: results.csv not found at {csv_path}")
            return None
        
        try:
            df = pd.read_csv(csv_path)
            print(f"✓ Loaded training results from {csv_path}")
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {list(df.columns)}")
            return df
        except Exception as e:
            print(f"Warning: Could not read results.csv: {e}")
            return None
    
    def plot_training_losses(self, output_dir):
        """
        Plot training loss curves (Box Loss + Classification Loss).
        
        Args:
            output_dir: Directory containing training results
            
        Returns:
            matplotlib figure
        """
        df = self.extract_results_csv(output_dir)
        if df is None:
            return None
        
        # Column name mappings (Ultralytics may use different naming conventions)
        column_mappings = {
            'epoch': ['epoch', 'Epoch'],
            'train_box_loss': ['train/box_loss', 'train_box_loss', 'box_loss'],
            'train_cls_loss': ['train/cls_loss', 'train_cls_loss', 'cls_loss'],
            'val_box_loss': ['val/box_loss', 'val_box_loss'],
            'val_cls_loss': ['val/cls_loss', 'val_cls_loss']
        }
        
        # Find actual column names
        def find_column(df, names):
            for name in names:
                if name in df.columns:
                    return name
            return None
        
        epoch_col = find_column(df, column_mappings['epoch'])
        train_box_col = find_column(df, column_mappings['train_box_loss'])
        train_cls_col = find_column(df, column_mappings['train_cls_loss'])
        val_box_col = find_column(df, column_mappings['val_box_loss'])
        val_cls_col = find_column(df, column_mappings['val_cls_loss'])
        
        if not epoch_col:
            print("Warning: Could not find epoch column")
            return None
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot Box Loss
        if train_box_col and val_box_col:
            axes[0].plot(df[epoch_col], df[train_box_col], label='Train Box Loss', linewidth=2, color='blue')
            axes[0].plot(df[epoch_col], df[val_box_col], label='Val Box Loss', linewidth=2, color='orange')
            axes[0].set_xlabel('Epoch', fontsize=12)
            axes[0].set_ylabel('Box Loss', fontsize=12)
            axes[0].set_title('Box Loss Curves', fontsize=14)
            axes[0].legend(fontsize=11)
            axes[0].grid(True, alpha=0.3)
        else:
            axes[0].text(0.5, 0.5, 'Box Loss data not available', 
                        ha='center', va='center', fontsize=12, transform=axes[0].transAxes)
            axes[0].set_title('Box Loss Curves', fontsize=14)
        
        # Plot Classification Loss
        if train_cls_col and val_cls_col:
            axes[1].plot(df[epoch_col], df[train_cls_col], label='Train Cls Loss', linewidth=2, color='green')
            axes[1].plot(df[epoch_col], df[val_cls_col], label='Val Cls Loss', linewidth=2, color='red')
            axes[1].set_xlabel('Epoch', fontsize=12)
            axes[1].set_ylabel('Classification Loss', fontsize=12)
            axes[1].set_title('Classification Loss Curves', fontsize=14)
            axes[1].legend(fontsize=11)
            axes[1].grid(True, alpha=0.3)
        else:
            axes[1].text(0.5, 0.5, 'Classification Loss data not available', 
                        ha='center', va='center', fontsize=12, transform=axes[1].transAxes)
            axes[1].set_title('Classification Loss Curves', fontsize=14)
        
        plt.tight_layout()
        return fig
    
    def plot_validation_mAP(self, output_dir):
        """
        Plot validation mAP curves (mAP@0.5 + mAP@0.5:0.95).
        
        Args:
            output_dir: Directory containing training results
            
        Returns:
            matplotlib figure
        """
        df = self.extract_results_csv(output_dir)
        if df is None:
            return None
        
        # Column name mappings
        column_mappings = {
            'epoch': ['epoch', 'Epoch'],
            'map50': ['metrics/mAP50(B)', 'metrics/mAP50', 'mAP50', 'map50'],
            'map50_95': ['metrics/mAP50-95(B)', 'metrics/mAP50-95', 'mAP50-95', 'map50-95']
        }
        
        def find_column(df, names):
            for name in names:
                if name in df.columns:
                    return name
            return None
        
        epoch_col = find_column(df, column_mappings['epoch'])
        map50_col = find_column(df, column_mappings['map50'])
        map50_95_col = find_column(df, column_mappings['map50_95'])
        
        if not epoch_col:
            print("Warning: Could not find epoch column")
            return None
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot mAP@0.5
        if map50_col:
            axes[0].plot(df[epoch_col], df[map50_col], label='mAP@0.5', linewidth=2, color='blue')
            axes[0].set_xlabel('Epoch', fontsize=12)
            axes[0].set_ylabel('mAP@0.5', fontsize=12)
            axes[0].set_title('mAP@0.5 Curve', fontsize=14)
            axes[0].legend(fontsize=11)
            axes[0].grid(True, alpha=0.3)
            axes[0].set_ylim(0, 1)
        else:
            axes[0].text(0.5, 0.5, 'mAP@0.5 data not available', 
                        ha='center', va='center', fontsize=12, transform=axes[0].transAxes)
            axes[0].set_title('mAP@0.5 Curve', fontsize=14)
        
        # Plot mAP@0.5:0.95
        if map50_95_col:
            axes[1].plot(df[epoch_col], df[map50_95_col], label='mAP@0.5:0.95', linewidth=2, color='green')
            axes[1].set_xlabel('Epoch', fontsize=12)
            axes[1].set_ylabel('mAP@0.5:0.95', fontsize=12)
            axes[1].set_title('mAP@0.5:0.95 Curve', fontsize=14)
            axes[1].legend(fontsize=11)
            axes[1].grid(True, alpha=0.3)
            axes[1].set_ylim(0, 1)
        else:
            axes[1].text(0.5, 0.5, 'mAP@0.5:0.95 data not available',
                         ha='center', va='center', fontsize=12, transform=axes[1].transAxes)
            axes[1].set_title('mAP@0.5:0.95 Curve', fontsize=14)

        plt.tight_layout()
        return fig
    
    def plot_confusion_matrix(self, output_dir=None):
        """
        Plot both normalized and unnormalized confusion matrices side-by-side.
        
        Args:
            output_dir: Directory containing training results
            
        Returns:
            matplotlib figure
        """
        import matplotlib.image as mpimg
        
        if output_dir is None:
            print("Warning: output_dir not provided")
            return None
            
        cm_path = Path(output_dir) / 'train' / 'confusion_matrix.png'
        cm_norm_path = Path(output_dir) / 'train' / 'confusion_matrix_normalized.png'
        
        if not cm_path.exists() and not cm_norm_path.exists():
            print("Warning: Confusion matrix images not found in train directory.")
            return None
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        
        # Plot Unnormalized
        if cm_path.exists():
            img_cm = mpimg.imread(cm_path)
            axes[0].imshow(img_cm)
            axes[0].set_title('Confusion Matrix (Unnormalized)', fontsize=14)
            axes[0].axis('off')
        else:
            axes[0].text(0.5, 0.5, 'Not Available', ha='center', va='center', fontsize=12)
            axes[0].set_title('Confusion Matrix (Unnormalized)', fontsize=14)
            axes[0].axis('off')

        # Plot Normalized
        if cm_norm_path.exists():
            img_cm_norm = mpimg.imread(cm_norm_path)
            axes[1].imshow(img_cm_norm)
            axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14)
            axes[1].axis('off')
        else:
            axes[1].text(0.5, 0.5, 'Not Available', ha='center', va='center', fontsize=12)
            axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14)
            axes[1].axis('off')
            
        plt.tight_layout()
        plt.show()
        return fig

    def plot_pr_curves_from_results(self, output_dir):
        """
        Plot Precision-Recall Curve from saved training results.
        Only displays the main BoxPR_curve.png.

        Args:
            output_dir: Directory containing training results

        Returns:
            matplotlib figure
        """
        import matplotlib.image as mpimg
        
        pr_curve_path = Path(output_dir) / 'train' / 'BoxPR_curve.png'
        
        if pr_curve_path.exists():
            img_pr = mpimg.imread(pr_curve_path)
            plt.figure(figsize=(10, 8))
            plt.imshow(img_pr)
            plt.title('Precision-Recall Curve (Box)', fontsize=14)
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            return plt.gcf()
        else:
            print(f"Warning: PR Curve image not found at {pr_curve_path}")
            return None

    def plot_training_results_summary(self, output_dir):
        """
        Plot the comprehensive results.png generated by Ultralytics during training.
        This includes Loss, mAP, Precision, Recall curves.
        
        Args:
            output_dir: Directory containing training results
            
        Returns:
            matplotlib figure
        """
        import matplotlib.image as mpimg
        
        results_png_path = Path(output_dir) / 'train' / 'results.png'
        
        if results_png_path.exists():
            img_results = mpimg.imread(results_png_path)
            plt.figure(figsize=(16, 10))
            plt.imshow(img_results)
            plt.title('Training Results Summary (Loss, mAP, P, R)', fontsize=14)
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            return plt.gcf()
        else:
            print(f"Warning: Results summary image not found at {results_png_path}")
            return None

print("✓ DetectionEvaluator defined")
print("\n✓ All YOLOv8 modules loaded successfully!")
